In [ ]:
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
file_path = "../data"

In [ ]:
df = pd.read_csv(f"{file_path}/ratings.csv").drop(labels="date", axis=1)

In [ ]:
spark = (
    SparkSession.builder
        .appName("system-recommendation")
        .getOrCreate()
)

In [ ]:
spark.sparkContext.setLogLevel('ERROR')

In [ ]:
spark_df = spark.read.csv(file_path, header=True, inferSchema=True)

In [ ]:
spark_df

In [ ]:
(train_df, test_df) = spark_df.randomSplit([0.8, 0.2])

In [ ]:
als_model = ALS(
    rank=10,
    maxIter=10,
    regParam=0.1,
    userCol="user_id",
    itemCol="item_id",
    ratingCol="rating",
    coldStartStrategy='drop'
)

In [ ]:
model = als_model.fit(train_df)

In [ ]:
test_predictions = model.transform(test_df)
test_predictions

In [ ]:
evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='rating',
    predictionCol='prediction'
)

In [ ]:
rmse = evaluator.evaluate(test_predictions)
print(f"RMSE: {rmse:.4f}")